In [17]:
from tqdm import tqdm
import pandas as pd
import requests
import time

tqdm.pandas()

df = pd.read_csv("../data/top_movies_cleaned_final.csv")

API_KEY = "862b087"

def get_imdb_data(title, year):
    url = "https://www.omdbapi.com/"
    params = {
        "apikey": API_KEY,
        "t": title,
        "y": int(year),
        "type": "movie"
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
    except Exception:
        return pd.Series([None, None, None])

    if data.get("Response") == "True":
        return pd.Series([
            data.get("imdbID"),
            data.get("imdbRating"),
            data.get("imdbVotes")
        ])

    return pd.Series([None, None, None])

# change these each day
start = 4000
end = 4480

part = df.iloc[start:end].copy()

part[["imdb_id", "imdb_rating", "imdb_votes"]] = part.progress_apply(
    lambda row: get_imdb_data(row["movie_name"], row["release_year"]),
    axis=1
)

part.to_csv(f"top_movies_with_imdb_{start}_{end}.csv", index=False)

100%|██████████| 480/480 [01:42<00:00,  4.70it/s]


In [18]:
df = pd.read_csv("top_movies_with_imdb_4000_4480.csv")

print("Total rows:", len(df))

print("Matched IMDb IDs:", df["imdb_id"].notna().sum())

print("Missing IMDb IDs:", df["imdb_id"].isna().sum())

print("Match rate:")

print(df["imdb_id"].notna().mean())

Total rows: 480
Matched IMDb IDs: 320
Missing IMDb IDs: 160
Match rate:
0.6666666666666666
